# 🔬 Agricultural Pest Image Retrieval System (CBIR) Demo on Kaggle
Notebook này hướng dẫn chi tiết cách chạy hệ thống truy vấn ảnh sâu bệnh nông nghiệp 2 giai đoạn (**Automatic Detection & Crop -> CLIP Search**) trên tập dữ liệu **IP102** trên Kaggle.

### ⚙️ Quy trình hoạt động của hệ thống:
1. **Giai đoạn 1 (Định vị & Crop)**: Sử dụng mô hình Open-World Object Detection (OW-OVD / YOLO-World) để tự động xác định vùng chứa sâu bệnh (bounding box) trong ảnh và cắt ra vùng đó (thêm 10% padding).
2. **Giai đoạn 2 (Trích xuất đặc trưng & Tìm kiếm)**: Sử dụng mô hình nền tảng **CLIP** của OpenAI để trích xuất vector đặc trưng visual (512 chiều) từ vùng ảnh đã cắt và tiến hành so sánh độ tương đồng Cosine (Cosine Similarity) với cơ sở dữ liệu Gallery.
3. **Màng lọc an toàn (HAUF Safeguard)**: Tính toán điểm bất định lạ ($P_u$) để cảnh báo người dùng nếu ảnh đầu vào chứa một loài sâu bệnh lạ không tồn tại trong cơ sở dữ liệu.

### ⚠️ Yêu cầu trước khi chạy:
1. Hãy chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** hoặc **GPU P100** trong phần settings của Kaggle (bên phải màn hình: *Accelerator -> GPU*).
2. Bật kết nối internet cho notebook (*Internet on*).

## 🛠️ Bước 1: Clone Repository từ GitHub
Tải mã nguồn mới nhất cùng với submodule `mmyolo` từ GitHub.

In [ ]:
# Khai báo thông tin repo
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Tải mmyolo vào thư mục third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi tương thích MMCV
Tự động cài đặt các thư viện cần thiết, PyTorch tương thích CUDA 12.1 và vá lỗi giới hạn phiên bản của MMCV để tránh xung đột khi import.

In [ ]:
print("-> 1. Thiết lập phiên bản PyTorch & Torchvision...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ wheel index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV vật lý trên đĩa cứng...")
import site
import os
import glob

def patch_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        new_content = content
        for old_ver in ["'2.1.0'", "'2.2.0'", '"2.1.0"', '"2.2.0"']:
            new_content = new_content.replace(f"mmcv_maximum_version = {old_ver}", "mmcv_maximum_version = '2.3.0'")
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"  [Vá lỗi] Đã cập nhật file: {file_path}")

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        patch_file(os.path.join(s_dir, pkg, "__init__.py"))

for init_file in glob.glob("**/mmyolo/__init__.py", recursive=True):
    patch_file(init_file)
for init_file in glob.glob("**/mmdet/__init__.py", recursive=True):
    patch_file(init_file)

paths_to_glob = [
    "/opt/conda/lib/python*/site-packages/mmdet/__init__.py",
    "/opt/conda/lib/python*/site-packages/mmyolo/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmdet/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmyolo/__init__.py"
]
for path_pattern in paths_to_glob:
    for init_file in glob.glob(path_pattern):
        patch_file(init_file)

print("\n-> 7. Kiểm tra import tất cả các package...")
import torch
import mmcv

# Bẫy phiên bản MMCV trong bộ nhớ trước khi import mmdet và mmyolo để vượt qua các xác thực phiên bản cứng đầu
real_mmcv_version = mmcv.__version__
mmcv.__version__ = '2.0.1'

import mmdet
import mmyolo

# Khôi phục phiên bản thật
mmcv.__version__ = real_mmcv_version

print(f"  - torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - mmcv: {mmcv.__version__}")
print(f"  - mmdet: {mmdet.__version__}")
print(f"  - mmyolo: {mmyolo.__version__}")
print("====== Khởi tạo môi trường hoàn tất! ======")

## 🗂️ Bước 3: Định vị Dataset & Sinh Đặc trưng phụ trợ (Auxiliary Embeddings)
Định vị dataset IP102 trên Kaggle và tự động sinh các file đặc trưng lớp gán nhãn, thuộc tính, và phân phối mẫu giống hệt lúc huấn luyện mô hình.

In [ ]:
import json
import torch
import numpy as np
import os
import glob
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Khởi tạo các thư mục
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải pretrain weights của YOLO-World làm nền tảng nếu cần
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights...")
    !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth

# 3. Định vị thư mục dataset IP102 trên Kaggle
dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break
if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")

# 4. Đọc nhãn các lớp từ train.json
ann_path = os.path.join(dataset_root, 'train.json')
if os.path.exists(ann_path):
    with open(ann_path, 'r') as f:
        coco_data = json.load(f)
    categories = sorted(coco_data['categories'], key=lambda x: x['id'])
    class_names = [cat['name'] for cat in categories]
else:
    print("-> Sử dụng danh sách class fallback...")
    from demo_app import IP102_CLASSES
    class_names = IP102_CLASSES[:102]

num_classes = len(class_names)
print(f"-> Tổng số lớp sâu bệnh: {num_classes}")

# 5. Lưu class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 6. Sinh class embeddings bằng CLIP
print("-> Đang trích xuất text embeddings bằng CLIP...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

# 7. Sinh task_att_1_embeddings.pth
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

# 8. Sinh mowod_distribution_sim1.pth
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Sinh file đặc trưng phụ trợ thành công! ======")

## Bước 4: Xây dựng cơ sở dữ liệu Gallery Index (Offline Index Builder)
Quét toàn bộ tập thư mục kiểm thử (Gallery) để định vị sâu bệnh và trích xuất vector đặc trưng bằng CLIP lưu trữ offline vào tệp chỉ mục. Thay đổi biến `CHECKPOINT_DIR` đến thư mục chứa các checkpoint của bạn trên Kaggle.

In [ ]:
# ĐƯỜNG DẪN CHECKPOINT CỦA BẠN TRÊN KAGGLE
CHECKPOINT_DIR = "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3"

# Cấu hình các tham số đánh giá cho Task 1
t1_config = "configs/open_world/mowod/custom/ip102_t1.py"
t1_checkpoint = os.path.join(CHECKPOINT_DIR, "best_coco_Current class AP50_epoch_5.pth")
gallery_folder = os.path.join(dataset_root, "test")
ann_file = os.path.join(dataset_root, "test.json")
output_index = "gallery_index_task1.pkl"

if not os.path.exists(t1_checkpoint):
    # Chế độ fallback sử dụng model YOLO-World gốc nếu chưa cấu hình checkpoint riêng
    t1_checkpoint = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
    print(f"⚠️ Không tìm thấy checkpoint tại {CHECKPOINT_DIR}. Dùng fallback weights gốc của YOLO-World.")

print("-> Đang khởi tạo CSDL đặc trưng ảnh mẫu (Gallery Index)... (Có thể mất 2-3 phút)")
!python build_gallery_index.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-dir "{gallery_folder}" \
    --ann-file "{ann_file}" \
    --output "{output_index}" \
    --device cuda:0

## Bước 5: Chạy thử tìm kiếm tương đồng trên một ảnh (Query Single Image)
Tự động chọn một ảnh ngẫu nhiên trong thư mục kiểm thử làm Query, định vị sâu bệnh bằng OW-OVD, tính toán điểm lạ HAUF, trích xuất đặc trưng CLIP và hiển thị trực quan top 5 ảnh tương đồng nhất được truy vấn từ Gallery.

In [ ]:
from IPython.display import Image, display
import glob

# Tự động tìm kiếm ảnh thực tế bất kỳ trong tập test để truy vấn tránh FileNotFoundError
test_images = glob.glob(os.path.join(dataset_root, "**/*.jpg"), recursive=True)
test_images = [f for f in test_images if "test" in f.replace("\\", "/")]
if test_images:
    query_image_path = test_images[0]
    print(f"-> Tự động tìm thấy ảnh query thực tế: {query_image_path}")
else:
    query_image_path = os.path.join(dataset_root, "test", "00001.jpg")
    print(f"-> Dùng fallback: {query_image_path}")

print("\n-> Đang chạy truy vấn ảnh sâu bệnh...")
!python retrieve.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-index "{output_index}" \
    --query-image "{query_image_path}" \
    --top-k 5 \
    --anomaly-thr 0.55 \
    --output query_result.jpg

# Hiển thị kết quả trực quan
if os.path.exists("query_result.jpg"):
    display(Image(filename="query_result.jpg"))
else:
    print("Error: Không tìm thấy ảnh kết quả 'query_result.jpg'.")

## Bước 6: Đánh giá hiệu năng của Hệ thống Truy vấn mở (Open-World Evaluator)
Cell này chạy đánh giá hàng loạt tập Query (`val.json`), tính toán các metric độ chính xác **Recall@1/5/10**, đồng thời đo lường khả năng lọc đối tượng lạ (Unseen) bằng **AUROC** và **FPR@TPR95** và vẽ biểu đồ ROC của bộ lọc HAUF Safeguard.

In [ ]:
from IPython.display import Markdown, display

report_file = "report_task1_retrieval.md"
print("-> Đang bắt đầu đánh giá hệ thống truy vấn trên tập validation...")

!python evaluate_retrieval.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --dataset-root "{dataset_root}" \
    --query-split val \
    --gallery-split test \
    --query-cache "query_cache_task1_demo.pkl" \
    --gallery-cache "gallery_cache_task1_demo.pkl" \
    --output-report "{report_file}" \
    --device cuda:0

# Hiển thị báo cáo kết quả cùng biểu đồ ROC
if os.path.exists(report_file):
    print(f"\n🔍 KẾT QUẢ ĐÁNH GIÁ CHI TIẾT (Đọc từ {report_file}):")
    display(Markdown(filename=report_file))
else:
    print(f"⚠️ Không tìm thấy báo cáo kết quả '{report_file}'. Vui lòng kiểm tra log lỗi ở trên.")